In [ ]:
from pathlib import Path
import pickle
from contextlib import contextmanager

import numpy as np
import pandas as pd
import pycountry
import re
import sklearn.utils

cwd = Path.cwd().resolve()

def find_release_root(start: Path) -> Path:
    sentinel_dirs = ('Step0_Data', 'Step9_FutureProjection')
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        nested_project = candidate / 'Global-solid-waste-prediction'
        if all((nested_project / name).is_dir() for name in sentinel_dirs):
            return nested_project
    raise FileNotFoundError('Could not locate Global-solid-waste-prediction repository root from the current working directory')

release_root = find_release_root(cwd)
shared_data_root = release_root / 'Step0_Data' / '1_Data'
output_dir = release_root / 'Step9_FutureProjection' / '2_Results'
step6_dir = release_root / 'Step7_ModelTraining' / '2_Results'
historical_path = release_root / 'Step8_HistoricalCompletion' / '2_Results' / '0_historical_panel_completed.csv'
ssp_basic_path = shared_data_root / 'SSP' / '1721734326790-ssp_basic_drivers_release_3.1_full.xlsx'
co2_path = shared_data_root / 'CO2' / 'PMSSPIE_05Feb20.csv'
output_dir.mkdir(parents=True, exist_ok=True)
ANCHOR_YEAR = 2024
TARGET_YEARS = list(range(2024, 2101))
WASTE_TYPES = ['AW', 'CDW', 'IW', 'MSW']
SCENARIO_FAMILY_MAP = {'SSP1': 'SSP1BL', 'SSP2': 'SSP2BL', 'SSP3': 'SSP3BL', 'SSP4': 'SSP4BL', 'SSP5': 'SSP5BL'}
TARGET_COUNTRY_FALLBACK = ['ATG', 'FSM', 'KIR', 'LCA', 'PRI', 'STP', 'SYC', 'TON', 'VCT']
WASTE_TARGET_COLS = ['AW_t', 'CDW_t', 'IW_t', 'MSW_t']
LOG_PROXY_BASES = ['NY.GDP.MKTP.PP.KD', 'SP.POP.TOTL', 'NY.GDP.PCAP.PP.KD', 'EN.POP.DNST', 'EN.GHG.CO2.MT.CE.AR5']
DRIVER_METRICS = ['SP.POP.TOTL', 'NY.GDP.MKTP.PP.KD', 'NY.GDP.PCAP.PP.KD', 'EN.POP.DNST']
DEFAULT_DRIVER_ALIGNMENT_THRESHOLDS = {'anchor_median_rel_diff_pct': 1e-6, 'trend_p90_abs_drift_pct': 5.0, 'identity_max_abs_diff': 1e-6}

@contextmanager
def _print_elapsed_time(*args, **kwargs):
    yield

if not hasattr(sklearn.utils, '_print_elapsed_time'):
    sklearn.utils._print_elapsed_time = _print_elapsed_time


In [ ]:
def year_columns(start=2024, end=2100):
    return [str(year) for year in range(start, end + 1)]

def extract_scenario(series):
    return series.astype(str).str.extract(r'(SSP[1-5])', expand=False)

def melt_ssp(frame, year_cols, value_name):
    out = frame.melt(id_vars=['Region', 'Scenario'], value_vars=year_cols, var_name='Year', value_name=value_name)
    out['Year'] = out['Year'].astype(int)
    out[value_name] = pd.to_numeric(out[value_name], errors='coerce')
    return out

def back_extrapolate_and_interpolate(df_long, value_col, mode='linear'):
    rows = []
    for (region, scenario), group in df_long.groupby(['Region', 'Scenario']):
        group = group.sort_values('Year').reset_index(drop=True)
        v2025 = float(group.loc[group['Year'] == 2025, value_col].iloc[0])
        v2030 = float(group.loc[group['Year'] == 2030, value_col].iloc[0])
        if abs(v2025) > 1e-10:
            rate = (v2030 / v2025) ** (1 / 5) - 1
            v2024 = v2025 / (1 + rate)
        else:
            v2024 = v2025
        group = pd.concat([group, pd.DataFrame([{'Region': region, 'Scenario': scenario, 'Year': 2024, value_col: v2024}])], ignore_index=True).sort_values('Year').reset_index(drop=True)
        year_to_value = dict(zip(group['Year'], group[value_col]))
        for year in TARGET_YEARS:
            if year in year_to_value:
                value = float(year_to_value[year])
            else:
                lower = max(y for y in year_to_value if y < year)
                upper = min(y for y in year_to_value if y > year)
                weight = (year - lower) / (upper - lower)
                lower_value = float(year_to_value[lower])
                upper_value = float(year_to_value[upper])
                if mode == 'log_linear' and lower_value > 0 and upper_value > 0:
                    value = float(np.exp(np.log(lower_value) + weight * (np.log(upper_value) - np.log(lower_value))))
                else:
                    value = float(lower_value + weight * (upper_value - lower_value))
            rows.append({'Region': region, 'Scenario': scenario, 'Year': year, value_col: value})
    return pd.DataFrame(rows)

def build_country_name_to_iso3():
    name_to_iso3 = {country.name: country.alpha_3 for country in pycountry.countries}
    name_to_iso3.update({
        'Bolivia': 'BOL',
        'Bolivia (Plurinational State of)': 'BOL',
        'Cabo Verde': 'CPV',
        'Czechia': 'CZE',
        'Congo': 'COG',
        'Democratic Republic of the Congo': 'COD',
        "Cote d'Ivoire": 'CIV',
        'Iran': 'IRN',
        'Iran (Islamic Republic of)': 'IRN',
        'Republic of Korea': 'KOR',
        'South Korea': 'KOR',
        'Laos': 'LAO',
        "Lao People's Democratic Republic": 'LAO',
        'Moldova': 'MDA',
        'Republic of Moldova': 'MDA',
        'Russian Federation': 'RUS',
        'Syria': 'SYR',
        'Syrian Arab Republic': 'SYR',
        'Tanzania': 'TZA',
        'United Republic of Tanzania': 'TZA',
        'Venezuela': 'VEN',
        'Venezuela (Bolivarian Republic of)': 'VEN',
        'Viet Nam': 'VNM',
        'Türkiye': 'TUR',
        'Turkey': 'TUR',
        'Micronesia': 'FSM',
        'Micronesia (Federated States of)': 'FSM',
        'Palestine': 'PSE',
        'China': 'CHN',
        'Hong Kong': 'HKG',
        'Macao': 'MAC',
        'Taiwan': 'TWN',
    })
    return name_to_iso3

AGGREGATE_REGION_PATTERNS = [r'\(R\d+\)', r'\bWorld\b', r'\bOECD\b', r'\bEuropean Union\b', r'\bEurope\b', r'\bAsia\b', r'\bAfrica\b', r'\bNorth America\b', r'\bLatin America\b', r'\bMiddle East\b', r'\bPacific OECD\b', r'\bOther OECD\b', r'\bOther Asia\b', r'\bRest of Asia\b', r'\bMiddle East & Africa\b', r'\bReforming Economies\b', r'\bChina\+\b', r'\bIndia\+\b']

def classify_region_reason(region):
    region_text = str(region)
    if region_text in {'Hong Kong', 'Macao', 'Taiwan'}:
        return 'non_mainland_china'
    if region_text == 'South Africa':
        return None
    for pattern in AGGREGATE_REGION_PATTERNS:
        if re.search(pattern, region_text):
            return 'aggregate_region'
    return None

def build_country_subset_audit(driver_df):
    base = driver_df[['Region', 'Country Code']].drop_duplicates().reset_index(drop=True)
    rows = []
    for _, row in base.iterrows():
        reason = classify_region_reason(row['Region'])
        included = True
        if reason is None and pd.isna(row['Country Code']):
            reason = 'unmapped_country'
            included = False
        elif reason is not None:
            included = False
        else:
            reason = 'included'
        rows.append({'Region': row['Region'], 'Country Code': row['Country Code'], 'included': included, 'reason': reason})
    return pd.DataFrame(rows).sort_values(['included', 'Region'], ascending=[False, True]).reset_index(drop=True)

def _zero_small(value, tol=1e-12):
    if pd.isna(value):
        return np.nan
    if abs(value) < tol:
        return 0.0
    return float(value)

def _relative_diff_pct(actual, expected):
    if pd.isna(actual) or pd.isna(expected):
        return np.nan
    scale = max(abs(float(expected)), 1e-12)
    return _zero_small((float(actual) - float(expected)) / scale * 100.0)

def _identity_expected_values(group):
    population = pd.Series(pd.to_numeric(group['SP.POP.TOTL'], errors='coerce'), index=group.index, dtype=float)
    gdp = pd.Series(pd.to_numeric(group['NY.GDP.MKTP.PP.KD'], errors='coerce'), index=group.index, dtype=float)
    area = pd.Series(pd.to_numeric(group['Area'], errors='coerce'), index=group.index, dtype=float)
    return {'NY.GDP.PCAP.PP.KD': gdp / population.replace(0, np.nan), 'EN.POP.DNST': population / area.replace(0, np.nan)}

def _compute_trend_drift_pct(group, metric):
    year_to_value = {}
    for year, value in group[['Year', metric]].itertuples(index=False, name=None):
        if pd.isna(year) or pd.isna(value):
            continue
        year_to_value[int(year)] = float(value)
    points = sorted((year, value) for year, value in year_to_value.items() if year >= 2024)
    if len(points) < 2:
        return np.nan
    base_year, base_value = points[0]
    edge_year, edge_value = points[1]
    tail_year, tail_value = points[-1]
    if edge_year == base_year or tail_year == base_year:
        return np.nan
    if min(base_value, edge_value, tail_value) <= 0:
        return np.nan
    edge_change = (np.log(edge_value) - np.log(base_value)) / (edge_year - base_year) * 100.0
    tail_change = (np.log(tail_value) - np.log(base_value)) / (tail_year - base_year) * 100.0
    return _zero_small(edge_change - tail_change, tol=1e-5)

def compute_driver_alignment_summary(driver_df, anchor_df):
    required_driver_cols = {'Country Code', 'Scenario', 'Year', *DRIVER_METRICS}
    missing_driver_cols = sorted(required_driver_cols.difference(driver_df.columns))
    if missing_driver_cols:
        raise ValueError(f'driver_df missing required columns: {missing_driver_cols}')
    required_anchor_cols = {'Country Code', 'Area', 'SP.POP.TOTL_anchor', 'NY.GDP.MKTP.PP.KD_anchor'}
    missing_anchor_cols = sorted(required_anchor_cols.difference(anchor_df.columns))
    if missing_anchor_cols:
        raise ValueError(f'anchor_df missing required columns: {missing_anchor_cols}')
    merged = driver_df.copy().merge(anchor_df[list(required_anchor_cols)], on='Country Code', how='left', validate='many_to_one')
    if bool(np.asarray(merged[['Area', 'SP.POP.TOTL_anchor', 'NY.GDP.MKTP.PP.KD_anchor']].isna()).any()):
        raise ValueError('anchor_df has missing matches for one or more driver rows')
    detail_rows = []
    for key, group in merged.groupby(['Country Code', 'Scenario'], dropna=False, sort=True):
        country_code, scenario = key
        group = group.sort_values('Year').reset_index(drop=True)
        anchor_year = group[group['Year'] == 2024]
        if anchor_year.empty:
            raise ValueError(f'missing 2024 anchor row for {country_code} / {scenario}')
        anchor_row = anchor_year.iloc[-1]
        identity_expected = _identity_expected_values(group)
        for metric in DRIVER_METRICS:
            expected_anchor = anchor_row[f'{metric}_anchor'] if metric in ['SP.POP.TOTL', 'NY.GDP.MKTP.PP.KD'] else identity_expected[metric].loc[anchor_row.name]
            actual_anchor = anchor_row[metric]
            identity_abs_diff = 0.0
            if metric in identity_expected:
                actual_series = pd.Series(pd.to_numeric(group[metric], errors='coerce'), index=group.index, dtype=float)
                identity_abs_diff = (actual_series - identity_expected[metric]).abs().max()
            detail_rows.append({'Country Code': country_code, 'Scenario': scenario, 'metric': metric, 'anchor_actual': float(actual_anchor), 'anchor_expected': float(expected_anchor), 'anchor_rel_diff_pct': _relative_diff_pct(actual_anchor, expected_anchor), 'trend_drift_pct': _compute_trend_drift_pct(group, metric), 'identity_max_abs_diff': _zero_small(identity_abs_diff)})
    detail = pd.DataFrame(detail_rows).sort_values(['Country Code', 'Scenario', 'metric']).reset_index(drop=True)
    summary = detail.groupby('metric', as_index=False).agg(anchor_median_rel_diff_pct=('anchor_rel_diff_pct', lambda s: _zero_small(s.dropna().median() if s.notna().any() else np.nan)), trend_p90_abs_drift_pct=('trend_drift_pct', lambda s: _zero_small(s.dropna().abs().quantile(0.9) if s.notna().any() else np.nan)), identity_max_abs_diff=('identity_max_abs_diff', lambda s: _zero_small(s.dropna().max() if s.notna().any() else np.nan)), group_count=('Country Code', 'count')).reset_index(drop=True)
    return pd.DataFrame(summary), detail

def validate_driver_alignment(driver_df, anchor_df, thresholds=None):
    thresholds = {**DEFAULT_DRIVER_ALIGNMENT_THRESHOLDS, **(thresholds or {})}
    summary, detail = compute_driver_alignment_summary(driver_df, anchor_df)
    checks = [('anchor drift too large', 'anchor_median_rel_diff_pct'), ('trend drift too large', 'trend_p90_abs_drift_pct'), ('identity drift too large', 'identity_max_abs_diff')]
    failures = []
    for label, column in checks:
        threshold = thresholds[column]
        failed_metrics = summary.loc[summary[column] > threshold, ['metric', column]]
        if not failed_metrics.empty:
            metrics_desc = ', '.join(f'{row.metric}={getattr(row, column):.6g}' for row in failed_metrics.itertuples(index=False))
            failures.append(f'{label}: {metrics_desc} > {threshold}')
    if failures:
        raise ValueError('; '.join(failures))
    return summary, detail


In [ ]:
def load_sspiamie_mt_co2():
    years = year_columns()
    frame = pd.read_csv(co2_path, usecols=['source', 'scenario', 'country', 'entity', 'unit', *years])
    frame = frame[(frame['entity'] == 'CO2') & (frame['source'] == 'SSPIAMIE') & (frame['unit'] == 'Mt')].copy()
    frame['scenario_family'] = frame['scenario'].astype(str).str.extract(r'(SSP[1-5]BL)', expand=False)
    return frame[frame['scenario_family'].notna()].reset_index(drop=True)

def summarize_filter(frame):
    return frame.groupby(['source', 'unit', 'scenario_family'], dropna=False).size().reset_index(name='rows').sort_values(['scenario_family', 'source', 'unit']).reset_index(drop=True)

def collapse_family_median(frame):
    years = year_columns()
    long_frame = frame.melt(id_vars=['country', 'scenario_family'], value_vars=years, var_name='Year', value_name='EN.GHG.CO2.MT.CE.AR5')
    long_frame['Year'] = long_frame['Year'].astype(int)
    long_frame['EN.GHG.CO2.MT.CE.AR5'] = pd.to_numeric(long_frame['EN.GHG.CO2.MT.CE.AR5'], errors='coerce')
    collapsed = long_frame.groupby(['country', 'scenario_family', 'Year'], as_index=False)['EN.GHG.CO2.MT.CE.AR5'].median().rename(columns={'country': 'Country Code'})
    collapsed['Proxy_EN.GHG.CO2.MT.CE.AR5_log1p'] = np.log1p(collapsed['EN.GHG.CO2.MT.CE.AR5'].clip(lower=0.0))
    return collapsed.sort_values(['Country Code', 'scenario_family', 'Year']).reset_index(drop=True)

def build_peer_ratio_pool(collapsed, hist_anchor):
    pool = collapsed.merge(hist_anchor[['Country Code', 'Region', 'IncomeGroup', 'EN.GHG.CO2.MT.CE.AR5']].rename(columns={'EN.GHG.CO2.MT.CE.AR5': 'hist_2024_co2'}), on='Country Code', how='left')
    pool = pool[pool['hist_2024_co2'].notna() & (pool['hist_2024_co2'] > 0)].copy()
    pool['ratio_vs_hist_2024'] = pool['EN.GHG.CO2.MT.CE.AR5'] / pool['hist_2024_co2']
    return pool

def build_fallback_frame(collapsed, hist_anchor):
    available = set(collapsed['Country Code'].dropna().astype(str).unique())
    missing = hist_anchor[hist_anchor['Country Code'].isin(TARGET_COUNTRY_FALLBACK) & ~hist_anchor['Country Code'].isin(available)].copy()
    if missing.empty:
        empty_frame = pd.DataFrame(columns=['Country Code', 'Scenario', 'scenario_family', 'Year', 'EN.GHG.CO2.MT.CE.AR5', 'Proxy_EN.GHG.CO2.MT.CE.AR5_log1p', 'fallback_level'])
        empty_summary = pd.DataFrame(columns=['Country Code', 'Scenario', 'scenario_family', 'fallback_level', 'peer_count', 'anchor_2024'])
        return empty_frame, empty_summary
    pool = build_peer_ratio_pool(collapsed, hist_anchor)
    ratio_region_income = pool.groupby(['scenario_family', 'Region', 'IncomeGroup', 'Year'], as_index=False)['ratio_vs_hist_2024'].median().rename(columns={'ratio_vs_hist_2024': 'peer_ratio'})
    ratio_region = pool.groupby(['scenario_family', 'Region', 'Year'], as_index=False)['ratio_vs_hist_2024'].median().rename(columns={'ratio_vs_hist_2024': 'peer_ratio'})
    peer_count_region_income = pool.groupby(['scenario_family', 'Region', 'IncomeGroup'], as_index=False)['Country Code'].nunique().rename(columns={'Country Code': 'peer_count_region_income'})
    peer_count_region = pool.groupby(['scenario_family', 'Region'], as_index=False)['Country Code'].nunique().rename(columns={'Country Code': 'peer_count_region'})
    frames = []
    summary_rows = []
    years = list(range(2024, 2101))
    for row in missing.to_dict(orient='records'):
        country_code = str(row['Country Code'])
        anchor = float(row['EN.GHG.CO2.MT.CE.AR5'])
        for scenario_name, family in SCENARIO_FAMILY_MAP.items():
            base = pd.DataFrame({'Country Code': [country_code] * len(years), 'Year': years, 'scenario_family': [family] * len(years)})
            merged = base.merge(ratio_region_income[(ratio_region_income['Region'] == row['Region']) & (ratio_region_income['IncomeGroup'] == row['IncomeGroup'])], on=['scenario_family', 'Year'], how='left')
            fallback_level = 'region+income'
            peer_count = peer_count_region_income[(peer_count_region_income['scenario_family'] == family) & (peer_count_region_income['Region'] == row['Region']) & (peer_count_region_income['IncomeGroup'] == row['IncomeGroup'])]['peer_count_region_income']
            if merged['peer_ratio'].isna().any():
                merged = base.merge(ratio_region[ratio_region['Region'] == row['Region']], on=['scenario_family', 'Year'], how='left')
                fallback_level = 'region-only'
                peer_count = peer_count_region[(peer_count_region['scenario_family'] == family) & (peer_count_region['Region'] == row['Region'])]['peer_count_region']
            if merged['peer_ratio'].isna().any():
                raise ValueError(f'Fallback ratio still missing for {country_code} / {family}')
            merged['EN.GHG.CO2.MT.CE.AR5'] = anchor * merged['peer_ratio']
            merged.loc[merged['Year'] == 2024, 'EN.GHG.CO2.MT.CE.AR5'] = anchor
            merged['Proxy_EN.GHG.CO2.MT.CE.AR5_log1p'] = np.log1p(merged['EN.GHG.CO2.MT.CE.AR5'].clip(lower=0.0))
            merged['Scenario'] = scenario_name
            merged['fallback_level'] = fallback_level
            frames.append(merged[['Country Code', 'Scenario', 'scenario_family', 'Year', 'EN.GHG.CO2.MT.CE.AR5', 'Proxy_EN.GHG.CO2.MT.CE.AR5_log1p', 'fallback_level']])
            summary_rows.append({'Country Code': country_code, 'Scenario': scenario_name, 'scenario_family': family, 'fallback_level': fallback_level, 'peer_count': int(peer_count.iloc[0]) if len(peer_count) else 0, 'anchor_2024': anchor})
    return pd.concat(frames, ignore_index=True), pd.DataFrame(summary_rows).sort_values(['Country Code', 'Scenario']).reset_index(drop=True)

def build_co2_driver_panel(driver_panel, hist_anchor):
    filtered = load_sspiamie_mt_co2()
    filter_summary = summarize_filter(filtered)
    collapsed = collapse_family_median(filtered)
    fallback_frame, fallback_summary = build_fallback_frame(collapsed, hist_anchor)
    native = collapsed.copy()
    native['Scenario'] = native['scenario_family'].map({value: key for key, value in SCENARIO_FAMILY_MAP.items()})
    native = native[native['Scenario'].notna()].copy()
    native['fallback_level'] = 'native'
    co2_panel = pd.concat([native[['Country Code', 'Scenario', 'scenario_family', 'Year', 'EN.GHG.CO2.MT.CE.AR5', 'Proxy_EN.GHG.CO2.MT.CE.AR5_log1p', 'fallback_level']], fallback_frame], ignore_index=True).sort_values(['Country Code', 'Scenario', 'Year']).reset_index(drop=True)
    driver_keys = driver_panel[['Country Code', 'Year', 'Scenario', 'IncomeGroup', 'Region']].drop_duplicates().copy()
    merged = driver_keys.merge(co2_panel, on=['Country Code', 'Year', 'Scenario'], how='left')
    if merged[['EN.GHG.CO2.MT.CE.AR5', 'Proxy_EN.GHG.CO2.MT.CE.AR5_log1p']].isna().any().any():
        missing = merged[merged['EN.GHG.CO2.MT.CE.AR5'].isna()][['Country Code', 'Year', 'Scenario']].head(20)
        raise ValueError(f'CO2 driver panel has missing rows after fallback: {missing.to_dict(orient="records")}')
    coverage_summary = merged.groupby(['Scenario', 'fallback_level'], dropna=False).size().reset_index(name='rows').sort_values(['Scenario', 'fallback_level']).reset_index(drop=True)
    return merged, filter_summary, coverage_summary, fallback_summary

def add_log_proxy_features(df, feature_columns):
    out = df.copy()
    for col in LOG_PROXY_BASES:
        if col not in out.columns:
            continue
        base = pd.to_numeric(out[col], errors='coerce')
        out[col] = base
        proxy_col = f'Proxy_{col}_log1p'
        if proxy_col in feature_columns:
            out[proxy_col] = np.log1p(base.clip(lower=0.0))
    return out

def assert_clean_feature_columns(columns):
    bad_columns = [col for col in columns if col.startswith('Is_') or '_x_' in col]
    if bad_columns:
        raise RuntimeError(f'Unexpected routed or flag columns in per-waste contract: {bad_columns}')

def prepare_corefeatures_from_wide(wide_df, core_feature_columns):
    wide_out = wide_df.copy()
    base_features = [c for c in wide_out.columns if c not in WASTE_TARGET_COLS]
    chunks = []
    for target in WASTE_TARGET_COLS:
        chunk = wide_out[base_features + [target]].copy()
        chunk['Waste_Generation_t'] = pd.to_numeric(chunk[target], errors='coerce')
        chunk = chunk.drop(columns=[target])
        chunk['WasteFlag'] = target.replace('_t', '')
        chunks.append(chunk)
    long_df = pd.concat(chunks, ignore_index=True).sort_values(['Country Code', 'Year', 'Scenario', 'WasteFlag']).reset_index(drop=True)
    assert_clean_feature_columns(core_feature_columns)
    long_df = add_log_proxy_features(long_df, core_feature_columns)
    missing = [c for c in core_feature_columns if c not in long_df.columns]
    if missing:
        raise RuntimeError(f'Missing required core features after proxy rebuild: {missing}')
    long_df[core_feature_columns] = long_df[core_feature_columns].apply(pd.to_numeric, errors='coerce')
    missing_counts = long_df[core_feature_columns].isna().sum()
    missing_counts = missing_counts[missing_counts > 0].sort_values(ascending=False)
    if not missing_counts.empty:
        raise RuntimeError(
            'Future raw contract has missing values after numeric coercion: '
            f'total_missing={int(missing_counts.sum())}; '
            f'by_column={missing_counts.to_dict()}'
        )
    return long_df


In [ ]:
historical = pd.read_csv(historical_path)
hist_anchor = historical[historical['Year'] == ANCHOR_YEAR].copy().drop_duplicates(subset=['Country Code'])
hist_anchor['Area'] = hist_anchor['SP.POP.TOTL'] / hist_anchor['EN.POP.DNST'].replace(0, np.nan)
hist_anchor['EN.GHG.CO2.MT.CE.AR5'] = pd.to_numeric(hist_anchor['EN.GHG.CO2.MT.CE.AR5'], errors='coerce')
ssp_basic = pd.read_excel(ssp_basic_path, sheet_name='data')
year_cols = [c for c in ssp_basic.columns if str(c).isdigit()]
pop = ssp_basic[(ssp_basic['Model'] == 'IIASA-WiC POP 2023') & (ssp_basic['Variable'] == 'Population')].copy()
pop['Scenario'] = extract_scenario(pop['Scenario'])
pop = pop.dropna(subset=['Scenario'])
pop_long = melt_ssp(pop[['Region', 'Scenario'] + year_cols], year_cols, 'Population')
gdp = ssp_basic[(ssp_basic['Model'] == 'IIASA GDP 2023') & (ssp_basic['Variable'] == 'GDP|PPP')].copy()
gdp['Scenario'] = extract_scenario(gdp['Scenario'])
gdp = gdp.dropna(subset=['Scenario'])
gdp_long = melt_ssp(gdp[['Region', 'Scenario'] + year_cols], year_cols, 'GDP_PPP')
print(f'Historical anchor rows: {hist_anchor.shape}')
print(f'SSP basic shape: {ssp_basic.shape}')
print(f'Population long: {pop_long.shape}')
print(f'GDP long: {gdp_long.shape}')


In [ ]:
pop_annual = back_extrapolate_and_interpolate(pop_long, 'Population', mode='log_linear')
gdp_annual = back_extrapolate_and_interpolate(gdp_long, 'GDP_PPP', mode='log_linear')
ssp = pop_annual.merge(gdp_annual, on=['Region', 'Scenario', 'Year'], how='inner')
name_to_iso3 = build_country_name_to_iso3()
ssp['Country Code'] = ssp['Region'].map(name_to_iso3)
audit = build_country_subset_audit(ssp[ssp['Year'] == ANCHOR_YEAR][['Region', 'Country Code']].copy())
anchor_countries = set(hist_anchor['Country Code'].dropna().astype(str).unique())
mask_missing_anchor = audit['included'] & ~audit['Country Code'].isin(anchor_countries)
audit.loc[mask_missing_anchor, 'included'] = False
audit.loc[mask_missing_anchor, 'reason'] = 'not_in_historical_182'
subset = audit[audit['included']].copy().sort_values('Country Code').reset_index(drop=True)
audit.to_csv(output_dir / '0_future_country_subset_audit.csv', index=False)
subset.to_csv(output_dir / '0_future_country_subset_included.csv', index=False)
audit[~audit['included']].to_csv(output_dir / '0_future_country_subset_excluded.csv', index=False)
print(f'Annualized SSP shape: {ssp.shape}')
print(f'Future eligible countries: {subset["Country Code"].nunique()}')
print(audit['reason'].value_counts(dropna=False))


In [ ]:
ssp = ssp[ssp['Country Code'].isin(subset['Country Code'])].copy()
anchor = hist_anchor[['Country Code', 'IncomeGroup', 'Region', 'SP.POP.TOTL', 'NY.GDP.MKTP.PP.KD', 'Area']].copy().rename(columns={'Region': 'Region_anchor', 'SP.POP.TOTL': 'SP.POP.TOTL_anchor', 'NY.GDP.MKTP.PP.KD': 'NY.GDP.MKTP.PP.KD_anchor'})
ssp = ssp.merge(anchor, on='Country Code', how='left')
anchor_2024 = ssp[ssp['Year'] == ANCHOR_YEAR].copy()
anchor_2024['pop_factor'] = anchor_2024['SP.POP.TOTL_anchor'] / anchor_2024['Population']
anchor_2024['gdp_factor'] = anchor_2024['NY.GDP.MKTP.PP.KD_anchor'] / anchor_2024['GDP_PPP']
corrections = anchor_2024[['Country Code', 'Scenario', 'pop_factor', 'gdp_factor']].copy()
ssp = ssp.merge(corrections, on=['Country Code', 'Scenario'], how='left')
ssp['SP.POP.TOTL'] = ssp['Population'] * ssp['pop_factor']
ssp['NY.GDP.MKTP.PP.KD'] = ssp['GDP_PPP'] * ssp['gdp_factor']
mask_2024 = ssp['Year'] == ANCHOR_YEAR
ssp.loc[mask_2024, 'SP.POP.TOTL'] = ssp.loc[mask_2024, 'SP.POP.TOTL_anchor']
ssp.loc[mask_2024, 'NY.GDP.MKTP.PP.KD'] = ssp.loc[mask_2024, 'NY.GDP.MKTP.PP.KD_anchor']
ssp['NY.GDP.PCAP.PP.KD'] = ssp['NY.GDP.MKTP.PP.KD'] / ssp['SP.POP.TOTL']
ssp['EN.POP.DNST'] = ssp['SP.POP.TOTL'] / ssp['Area']
driver_panel = ssp[['Country Code', 'Year', 'Scenario', 'IncomeGroup', 'Region_anchor', 'SP.POP.TOTL', 'NY.GDP.MKTP.PP.KD', 'NY.GDP.PCAP.PP.KD', 'EN.POP.DNST']].copy().rename(columns={'Region_anchor': 'Region'})
alignment_anchor = anchor[['Country Code', 'SP.POP.TOTL_anchor', 'NY.GDP.MKTP.PP.KD_anchor', 'Area']].copy()
driver_alignment_summary, driver_alignment_detail = validate_driver_alignment(driver_panel[['Country Code', 'Scenario', 'Year', 'SP.POP.TOTL', 'NY.GDP.MKTP.PP.KD', 'NY.GDP.PCAP.PP.KD', 'EN.POP.DNST']].copy(), alignment_anchor)
driver_keys = driver_panel[['Country Code', 'Year', 'Scenario', 'IncomeGroup', 'Region']].drop_duplicates().copy()
co2_driver_panel, filter_summary, coverage_summary, fallback_summary = build_co2_driver_panel(driver_keys, hist_anchor[['Country Code', 'Region', 'IncomeGroup', 'EN.GHG.CO2.MT.CE.AR5']].copy())
co2_driver_subset = co2_driver_panel[['Country Code', 'Year', 'Scenario', 'EN.GHG.CO2.MT.CE.AR5']].drop_duplicates()
future_driver_panel = driver_panel.merge(co2_driver_subset, on=['Country Code', 'Year', 'Scenario'], how='left').sort_values(['Country Code', 'Scenario', 'Year']).reset_index(drop=True)
for waste in WASTE_TYPES:
    future_driver_panel[f'{waste}_t'] = np.nan
driver_alignment_summary.to_csv(output_dir / '0_driver_alignment_summary.csv', index=False)
driver_alignment_detail.to_csv(output_dir / '0_driver_alignment_detail.csv', index=False)
co2_driver_panel.to_csv(output_dir / '0_future_co2_driver_panel.csv', index=False)
filter_summary.to_csv(output_dir / '0_co2_filter_summary.csv', index=False)
coverage_summary.to_csv(output_dir / '0_co2_coverage_summary.csv', index=False)
fallback_summary.to_csv(output_dir / '0_co2_fallback_summary.csv', index=False)
future_driver_panel.to_csv(output_dir / '0_future_driver_panel.csv', index=False)
print(f'Future driver panel shape: {future_driver_panel.shape}')
print(f'CO2 panel shape: {co2_driver_panel.shape}')
print(f'Countries: {future_driver_panel["Country Code"].nunique()}')


In [ ]:
base_contract = pd.read_csv(step6_dir / '0_feature_contract_raw.csv')['feature'].astype(str).tolist()
processed_contract = pd.read_csv(step6_dir / '0_feature_contract_processed.csv')['feature'].astype(str).tolist()
assert_clean_feature_columns(base_contract)
assert_clean_feature_columns(processed_contract)
future_driver_panel = pd.read_csv(output_dir / '0_future_driver_panel.csv')
long_df = prepare_corefeatures_from_wide(future_driver_panel, base_contract)
future_raw = long_df[['Country Code', 'Year', 'Scenario', 'WasteFlag'] + base_contract].copy()
future_raw.to_csv(output_dir / '0_future_feature_matrix_raw.csv', index=False)
print(f'Long table shape: {long_df.shape}')
print(f'Future raw feature matrix shape: {future_raw.shape}')
print(f'Scenario non-null: {int(future_raw["Scenario"].notna().sum())}/{len(future_raw)}')


In [ ]:
future_raw = pd.read_csv(output_dir / '0_future_feature_matrix_raw.csv')
for waste_type in WASTE_TYPES:
    part = future_raw[future_raw['WasteFlag'] == waste_type].copy().reset_index(drop=True)
    with open(step6_dir / f'0_preprocessing_pipeline_{waste_type}.pkl', 'rb') as handle:
        pipeline = pickle.load(handle)
    transformed = pipeline.transform(part[base_contract].copy())
    if isinstance(transformed, pd.DataFrame):
        if list(transformed.columns) != processed_contract:
            raise RuntimeError(f'Future transformed contract drift for {waste_type}: dataframe columns mismatch')
        transformed_df = transformed[processed_contract].copy()
    else:
        transformed_array = np.asarray(transformed)
        if transformed_array.ndim != 2:
            raise RuntimeError(f'Future transformed contract drift for {waste_type}: expected 2D output, got ndim={transformed_array.ndim}')
        if transformed_array.shape[1] != len(processed_contract):
            raise RuntimeError(
                f'Future transformed contract drift for {waste_type}: '
                f'expected {len(processed_contract)} columns, got {transformed_array.shape[1]}'
            )
        transformed_df = pd.DataFrame(transformed_array, columns=processed_contract)
    out = pd.concat([part[['Country Code', 'Year', 'Scenario', 'WasteFlag']], transformed_df], axis=1)
    out.to_csv(output_dir / f'0_future_features_transformed_{waste_type}.csv', index=False)
sorted(path.name for path in output_dir.glob('0_*.csv'))
